# Modeling và Evaluation — Customer Purchase Propensity

Notebook trình bày luồng chính: baseline recency, Logistic Regression và LightGBM trên 18 feature; sau đó đọc kết quả pipeline POC với chu kỳ đã cố định. Các thí nghiệm feature, calibration và độ nhạy chi tiết nằm trong [khu vực nghiên cứu](research/).

Target là có bất kỳ basket trong 7 ngày sau cutoff. Household có thể xuất hiện nhiều cutoff; kết quả không đo tác động của coupon. Tập test từng được xem trong quá trình phát triển.

## 1. Nạp train và validation

Bộ đọc kiểm tra hợp đồng dữ liệu và giới hạn cột. Các cờ bất thường, household ID, cutoff và target không phải feature. Tại bước huấn luyện mô hình ban đầu này không đọc test.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss,log_loss,roc_auc_score
ROOT=Path.cwd()
if ROOT.name=='notebooks':ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
if (ROOT/'.deps').is_dir():sys.path.insert(0,str(ROOT/'.deps'))
from propensity_v2.prepared_data import load_prepared_split
from lightgbm import LGBMClassifier
DATA=ROOT/'artifacts'/'data_preparation'
train,features=load_prepared_split(DATA,'train')
validation,vfeatures=load_prepared_split(DATA,'validation')
assert features==vfeatures and len(features)==18
assert train.snapshot_day.max()+7<validation.snapshot_day.min()
print('Train:',len(train),'| Validation:',len(validation),'| Feature:',len(features))

Train: 129834 | Validation: 27480 | Feature: 18


## 2. Baseline và hai mô hình ban đầu

Baseline tính tỷ lệ mua của các nhóm recency trên train, với fallback về tỷ lệ train chung nếu nhóm rỗng. Logistic Regression chuẩn hóa chỉ bằng train; LightGBM dùng cấu hình khởi đầu. Cả hai dùng 18 feature, fit trên cutoff train đến 537. Các model này là so sánh thăm dò, chưa phải pipeline POC cuối.

In [2]:
labels=['0–7','8–14','15–30','31–60','61–90','91+']
bins=[-1,7,14,30,60,90,np.inf]
train_group=pd.cut(train.recency_days,bins=bins,labels=labels)
val_group=pd.cut(validation.recency_days,bins=bins,labels=labels)
rates=train.assign(group=train_group).groupby('group',observed=False).target_any_basket.mean()
rates=rates.fillna(train.target_any_basket.mean())
probabilities={'Baseline recency':val_group.map(rates).astype(float).to_numpy()}
logistic=make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000,random_state=42))
lightgbm=LGBMClassifier(n_estimators=200,learning_rate=.05,num_leaves=15,min_child_samples=100,random_state=42,verbosity=-1,n_jobs=2)
for name,model in [('Logistic Regression',logistic),('LightGBM',lightgbm)]:
    model.fit(train[features],train.target_any_basket)
    probabilities[name]=model.predict_proba(validation[features])[:,1]
assert all(np.isfinite(p).all() and ((p>=0)&(p<=1)).all() for p in probabilities.values())
print('Đã huấn luyện và dự báo trên validation.')

Đã huấn luyện và dự báo trên validation.


## 3. So sánh trên validation

Brier/log loss càng thấp càng tốt; ROC-AUC đo xếp hạng. Lift tính top 10% riêng ở từng cutoff và trung bình không trọng số. Nếu đồng điểm tại ngưỡng, tính kỳ vọng chọn ngẫu nhiên trong nhóm đồng điểm; không dùng target để phá hòa. Lift không phải tác động can thiệp.

In [3]:
def weekly_lift(frame,p):
    values=[]
    y=frame.target_any_basket.to_numpy()
    for ix in frame.groupby('snapshot_day').indices.values():
        target=y[ix];q=p[ix];k=int(np.ceil(.1*len(ix)))
        threshold=np.partition(q,len(q)-k)[len(q)-k];higher=q>threshold;ties=q==threshold
        precision=(target[higher].sum()+(k-higher.sum())*target[ties].mean())/k
        values.append(precision/target.mean() if target.mean() else np.nan)
    return np.nanmean(values)
comparison=[]
for name,p in probabilities.items():
    comparison.append({'model':name,'brier':brier_score_loss(validation.target_any_basket,p),
                       'log_loss':log_loss(validation.target_any_basket,p,labels=[0,1]),
                       'roc_auc':roc_auc_score(validation.target_any_basket,p),
                       'weekly_lift_mean':weekly_lift(validation,p)})
comparison=pd.DataFrame(comparison).set_index('model');display(comparison)

,brier,log_loss,roc_auc,weekly_lift_mean
model,,,,
Baseline recency,0.199591,0.584073,0.726843,1.313018
Logistic Regression,0.166371,0.501671,0.833017,1.777743
LightGBM,0.164389,0.495405,0.836662,1.786582


## 4. Feature chu kỳ và rolling backtest

Thí nghiệm bổ sung median/std khoảng cách giữa các ngày mua trong 180 ngày, tỷ lệ recency/median và cờ đủ dữ liệu. Trong bốn fold train, Brier và log loss giảm ở cả bốn fold với cả Logistic và LightGBM, nhưng lift không tăng nhất quán. Các nghiên cứu không chứng minh chu kỳ sẽ cải thiện trên dữ liệu tương lai.

Bảng sau đọc kết quả đã lưu; để tái tạo các fold, chạy [rolling backtest](research/rolling_backtest_chu_ky_mua.ipynb). Đây là bảng kết quả nghiên cứu, không fit lại mô hình trong cell này.

In [4]:
path=ROOT/'artifacts'/'research'/'cadence_rolling_backtest'/'fold_feature_differences.csv'
if not path.exists():raise FileNotFoundError('Chạy research/rolling_backtest_chu_ky_mua.ipynb để tạo kết quả')
folds=pd.read_csv(path)
view=folds.loc[folds.train_scenario.eq('toàn bộ')&folds.eval_scenario.eq('toàn bộ'),
               ['fold','model','delta_brier','delta_log_loss','delta_roc_auc','delta_weekly_lift']]
display(view)

,fold,model,delta_brier,delta_log_loss,delta_roc_auc,delta_weekly_lift
0,1,Logistic Regression,-0.000913,-0.002397,0.001486,0.003847
2,1,LightGBM,-0.000814,-0.002120,0.001509,0.005124
8,2,Logistic Regression,-0.000631,-0.001968,0.000882,-0.000047
10,2,LightGBM,-0.000813,-0.002228,0.001659,0.006365
16,3,Logistic Regression,-0.000743,-0.002313,0.001114,0.000000
18,3,LightGBM,-0.000459,-0.001023,0.001021,0.001026
24,4,Logistic Regression,-0.000893,-0.003267,0.001429,0.003678
26,4,LightGBM,-0.000400,-0.000918,0.000748,-0.012535


## 5. Pipeline POC cuối và kết quả đã lưu

Ứng viên được chốt là LightGBM 18 feature + chu kỳ; model fit đến cutoff 439, sigmoid fit 453–481, lựa chọn calibration 495–537. Theo quy tắc đặt trước, giữ xác suất thô. Bảng đọc kết quả đã lưu của cả ứng viên và đối chứng 18 feature từ [nghiên cứu calibration](research/calibration_va_danh_gia_poc.ipynb).

**Khác biệt phạm vi:** các model ở mục 2 fit đến cutoff 537; pipeline POC ở mục này fit đến 439. Không so trực tiếp như thể là cùng model. Test trong bảng đã từng được xem nên chỉ là kết quả tham khảo, không phải holdout hoàn toàn độc lập.

In [5]:
poc_path=ROOT/'artifacts'/'calibration_final_poc'/'metrics.csv'
if not poc_path.exists():raise FileNotFoundError('Chạy research/calibration_va_danh_gia_poc.ipynb để tạo kết quả POC')
poc_metrics=pd.read_csv(poc_path)
display(poc_metrics)

,split,feature_set,rows,brier,log_loss,roc_auc,weekly_lift_mean
0,validation,18 feature,27480,0.165599,0.498827,0.834966,1.785994
1,validation,18 + chu kỳ,27480,0.165523,0.498580,0.835198,1.791388
2,test,18 feature,24994,0.168325,0.505472,0.829951,1.822417
3,test,18 + chu kỳ,24994,0.168401,0.505456,0.829773,1.818039


## 6. Kết luận và bước scoring

Các model đã thể hiện tín hiệu dự báo so với baseline recency. Lợi ích bổ sung của chu kỳ nhỏ và chưa ổn định cho lift; hai phương án POC có kết quả gần nhau. Không suy ra uplift hay ROI từ propensity. Tiếp theo dùng [notebook scoring](deployment_gia_lap_propensity.ipynb); test đã xem và việc không có nhãn sau ngày 711 phải được giữ trong báo cáo.